# 00 · RAG system map — when a model needs external evidence

> **Core lesson:** all cells execute without external services. Read the *predict → run → explain* questions before each experiment.


## Business situation and mental model

A support team's **fictional** return rules change often. A language model's learned parameters might not contain today's rule. In Retrieval-Augmented Generation (RAG), we **index trusted source documents**, retrieve relevant fragments for each user question, and supply them as **context** to a generative model.

```text
DOCUMENT SIDE (changes when policies change)            QUESTION SIDE (on every query)
Policies -> validate -> chunks -> vector index -----┐    User question
                                                    ├──> retrieve -> evidence + source ID
                                                    └──> grounded prompt -> LLM -> cited answer
```

**Predict:** If a return deadline changes from 14 to 21 days, does RAG require LLM retraining? **No**: reindex the changed document and validate retrieval. Fine-tuning can affect style/tasks, but cannot replace a reliable fresh source.

**Two modes in this curriculum:** (1) offline sparse TF-IDF retrieval + verbatim evidence preview; (2) optional REAL LLM generation using local Ollama in notebook 03. The first is *not* disguised as generative inference.


In [1]:
from pathlib import Path
from RAG.src.core import load_documents, chunk_documents, Retriever
DATA = Path("RAG/data/support_policies.jsonl")
docs = load_documents(DATA)
print("Documents:", len(docs))
print("First ID:", docs[0].doc_id)
print("First title:", docs[0].title)
print("Source provenance:", docs[0].source)


Documents: 10
First ID: shipping-india
First title: Domestic shipping
Source provenance: Fictional / shipping-india


## One complete retrieval round trip

The source includes an identifier that survives every stage; without provenance, a fluent answer is not auditable. Trace **question → matched document → citation**.


In [2]:
chunks = chunk_documents(docs, size=80, overlap=15)
retriever = Retriever(chunks)
question = "How do I reset my password?"
hits = retriever.search(question, k=2)
for h in hits:
    print(f"{h.chunk.doc_id} | cosine={h.score:.3f} | chunk={h.chunk.chunk_id}")
assert hits[0].chunk.doc_id == "account"


account | cosine=0.369 | chunk=account:0000


## Thought experiment (change → effect)

- Change the **question wording** to "forgot my sign-in credentials." Does word-based retrieval still rank the account policy first? Why might it fail?
- Change the **document content**, rebuild the index, rerun the same question. Explain why this is not updating LLM weights.
- Ask for the CEO's birthday. There is no supporting document, so the system must **abstain** rather than invent a fact.

**Student outcome:** Can you trace one answer back to a document ID, and explain which component selected it?


In [3]:
from RAG.src.core import run_query
for question in ("What is the API requests per minute limit?",
                 "What is the CEO's birthday?"):
    result = run_query(question, retriever)
    print(question)
    print(result["answer"][:200])
    print("Citations:", [s["doc_id"] for s in result["sources"]])
    print("---")


What is the API requests per minute limit?
Evidence preview (not LLM-generated): The free developer plan allows one hundred API requests per minute. The paid developer plan allows one thousand requests per minute. A request over the limit rece
Citations: ['api']
---
What is the CEO's birthday?
I do not know from the provided documents.
Citations: []
---


## Engineering crosswalk

| Student concept | System responsibility | Production consideration |
|---|---|---|
| Document | authoritative evidence | owner, timestamps, ACLs |
| Chunk | retrieval granularity | tables, overlap, context loss |
| Vectorizer | searchable representation | embedding model versions |
| Retriever | evidence selection | hybrid search, reranking |
| Generator | answer from context | prompt safety, grounded citations |
| Evaluator | correctness evidence | holdout dataset, human review |

Continue to **01_ingest_and_chunk**. See [the architecture guide](../docs/architecture.md).
